# Sentinel-2 MSI RGB Composite

**Source**: `grdl/example/IO/eo/view_sentinel2.py`

## Learning Objectives

- Load Sentinel-2 L1C/L2A products
- Extract RGB bands (B04, B03, B02)
- Create true-color composite

## Data Setup

**Path Configuration**: Set path to a Sentinel-2 .SAFE directory.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
# Configuration
SENTINEL2_PATH = '/path/to/your/S2A_MSIL1C_*.SAFE'
CHIP_SIZE = 2048

print(f"Sentinel-2 product: {SENTINEL2_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from grdl.IO import get_reader
from grdl.contrast import PercentileStretch

In [ ]:
# Load Sentinel-2 RGB bands
with get_reader('sentinel2', SENTINEL2_PATH) as reader:
    # Read RGB bands: B04 (red), B03 (green), B02 (blue)
    # Note: Sentinel-2 band numbering starts at 1
    red = reader.read_chip(0, CHIP_SIZE, 0, CHIP_SIZE, bands=[3])  # B04
    green = reader.read_chip(0, CHIP_SIZE, 0, CHIP_SIZE, bands=[2])  # B03
    blue = reader.read_chip(0, CHIP_SIZE, 0, CHIP_SIZE, bands=[1])  # B02
    
    print(f"Red band (B04) shape: {red.shape}")
    print(f"Green band (B03) shape: {green.shape}")
    print(f"Blue band (B02) shape: {blue.shape}")

In [ ]:
# Stack RGB channels
rgb = np.stack([red, green, blue], axis=-1)
print(f"RGB composite shape: {rgb.shape}")

# Apply percentile stretch for display
stretch = PercentileStretch(pmin=2, pmax=98)
rgb_display = stretch.apply(rgb)

print(f"Display range: [{rgb_display.min():.3f}, {rgb_display.max():.3f}]")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(rgb_display)
ax.set_title('Sentinel-2 True-Color Composite (B04/B03/B02)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cleanup memory
import gc

del red, green, blue, rgb, rgb_display
gc.collect()
print("✓ Memory cleanup complete")

## Summary

**GRDL patterns**:
- ✅ `get_reader('sentinel2', path)` — Sentinel-2 reader
- ✅ Band-selective reading — extract specific spectral bands
- ✅ `PercentileStretch` — multi-band contrast enhancement